# 실험 012 — 011 기반 4-fold Cross-validation / 16회 학습

루트에서 저장하고 위에서부터 전체 실행합니다. 기존 jihokamp 커널을 사용합니다.

| Fold | 네 증강 공통 학습 seed | 학습 횟수 |
|---|---|---:|
| 1 | 42 | 4 |
| 2 | 43 | 4 |
| 3 | 44 | 4 |
| 4 | 45 | 4 |

- 층화 4-fold를 split seed 42로 한 번 생성합니다. 4회 무작위 holdout이 아닙니다.
- 매 fold에서 정사각형 / 직사각형 허용 / 정사각형+가림 / 정사각형+원근을 새로 학습합니다.
- 011의 192 입력·크롭·증강·학습률·temperature를 유지합니다. 이전 가중치를 재사용하지 않습니다.
- 각 사진을 학습하지 않은 해당 fold 모델로 평가한 OOF 확률과 지표를 저장합니다.
- 전체 16모델은 약 64M이므로 한 앙상블로 사용하지 않습니다. 제출은 선택한 한 fold의 네 모델(약 16M)입니다.
- 제출 fold 기본값은 1입니다. 검증 점수로 자동 선택하지 않습니다.
- 기본 T0.3 / 전체 T0.5 / T0.3 TTA CSV와 각 대응 ZIP을 생성하고 CPU 재현을 검사합니다.
- 기존 011 및 #49·#80 코드·산출물은 보존합니다.


## 실행 준비

루트에 baseline_test6.ipynb, train/6e, train/it, test, sample_submission.csv를 준비합니다.
결과는 실험결과/012_4fold_192크롭/에 저장합니다. 기존 학습 결과가 있으면 덮어쓰지 않고 중단합니다.
4개 증강 × 4-fold, 총 16회 새 학습하므로 011보다 시간이 늘어납니다. 각 모델 최대 50 epoch이며 중간 결과를 즉시 저장합니다.
학습 후 제출 fold 선택 셀에서 1~4 중 하나를 선택합니다. 다른 fold를 내보낼 때는 해당 셀부터 아래만 재실행합니다.


## 모듈 불러오기 & 시드 고정

재현성을 위해 모든 난수 시드를 고정합니다. 시드가 고정되어 있어야 "내가 바꾼 것 때문에 점수가 오른 것인지, 운이 좋았던 것인지"를 구분할 수 있습니다.

In [ ]:
import csv
import hashlib
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from PIL import Image


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 42
BASE_DIR = Path.cwd().resolve()
NOTEBOOK_PATH = BASE_DIR / 'baseline_test6.ipynb'
assert NOTEBOOK_PATH.is_file(), f'루트에서 노트북을 저장한 뒤 실행하세요: {NOTEBOOK_PATH}'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(SEED)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
	print(torch.cuda.get_device_name())


## 하이퍼 파라미터

In [ ]:
params = {
	'epoch': 50,
	'early_stopping': True,
	'min_epochs': 5,
	'patience': 5,
	'batch_size': 32,
	'learning_rates': (0.001, 0.0003, 0.0001),
	'workers': 0,
	'num_show_img': 5,
}


In [ ]:
class_names = {0: '6e', 1: 'it'}
N_SPLITS = 4
FOLD_SEEDS = (42, 43, 44, 45)
AUGMENTATIONS = ('square', 'rectangle', 'square_erasing', 'square_perspective')
assert len(FOLD_SEEDS) == N_SPLITS
RESULT_DIR = BASE_DIR / '실험결과' / '012_4fold_192크롭'
CODE_DIR = RESULT_DIR / '코드'
assert not (RESULT_DIR / 'runs.csv').exists(), f'기존 결과를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not list(RESULT_DIR.glob('fold_*/checkpoints/*.pt')), f'기존 체크포인트가 있습니다: {RESULT_DIR}'
CODE_DIR.mkdir(parents=True, exist_ok=True)
for fold in range(1, N_SPLITS + 1):
	for directory in ('checkpoints', 'epochs', 'graphs'):
		(RESULT_DIR / f'fold_{fold}' / directory).mkdir(parents=True, exist_ok=True)
print('결과 위치:', RESULT_DIR)
print('학습 횟수:', N_SPLITS * len(AUGMENTATIONS))


## 모델·크롭 정의 (`model_test6.py` 생성)

다른 실험과 파일이 섞이지 않도록 별도 모듈을 생성합니다. 제출 ZIP에서는 model.py라는 이름으로 복사합니다.


In [ ]:
%%writefile model_test6.py
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as F

IMG_SIZE = 192
MIN_CROP_SIZE = 96
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PERSPECTIVE_FILL = tuple(round(value * 255) for value in NORM_MEAN)
TEMPERATURE_RULES = ("T0.3", "T0.5")


class RandomPhotoCrop:
	def __init__(self, square):
		self.square = square

	def __call__(self, image):
		width, height = image.size
		assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작은 원본입니다: {image.size}"
		crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		if self.square:
			crop_height = crop_width
		else:
			crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
			while (crop_width / crop_height < 0.75) or (crop_width / crop_height > 4 / 3):
				crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
				crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		top = int(torch.randint(height - crop_height + 1, ()).item())
		left = int(torch.randint(width - crop_width + 1, ()).item())
		return F.crop(image, top, left, crop_height, crop_width)


def center_crop(image):
	width, height = image.size
	assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작아서 패딩 없이 자를 수 없습니다: {image.size}"
	return F.center_crop(image, [IMG_SIZE, IMG_SIZE])


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(center_crop),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs, temperature_rule="T0.3"):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 4)
	assert bool(torch.isfinite(member_probs).all()), "모델 확률에 비유한 값이 있습니다."
	assert bool(((member_probs >= 0) & (member_probs <= 1)).all()), "モデル確率が範囲外です。"
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	middle_mean = member_probs.sort(dim=1).values[:, 1:3].mean(dim=1, keepdim=True)
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), middle_mean)
	if temperature_rule == "T0.3":
		temperatures = torch.where(unanimous, 0.3, 0.7)
	elif temperature_rule == "T0.5":
		temperatures = 0.5
	else:
		raise ValueError(f"알 수 없는 temperature 규칙: {temperature_rule}")
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(pretrained=False):
	weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
	net = models.efficientnet_b0(weights=weights)
	net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	return net


class EfficientNetEnsemble(nn.Module):
	def __init__(self, state_dicts):
		super().__init__()
		assert len(state_dicts) == 4, "앙상블 가중치 4개가 필요합니다."
		self.models = nn.ModuleList()
		for state_dict in state_dicts:
			model = build_model()
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs):
		return torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)

	def forward(self, inputs, temperature_rule="T0.3"):
		return combine_probabilities(self.member_probabilities(inputs), temperature_rule)


In [ ]:
import model_test6 as model_module
importlib.invalidate_caches()
importlib.reload(model_module)
assert Path(model_module.__file__).resolve() == BASE_DIR / 'model_test6.py'
model = model_module.build_model()
print('EfficientNet-B0 / input:', model_module.IMG_SIZE)


### 파라미터 카운터 (20M 제한 확인)

앙상블을 하거나 사전학습 백본을 쓰는 경우 **모든 모델의 파라미터를 합산**한 값이 20,000,000 이하여야 합니다.

In [ ]:
PARAM_LIMIT = 20_000_000
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f'1모델 {n_params:,} / 제출용 4모델 {4 * n_params:,} / 제한 {PARAM_LIMIT:,}')
print(f'총 학습 모델 수 {N_SPLITS * len(AUGMENTATIONS)}개. 전체를 한 앙상블로 사용하지 않습니다.')
assert 4 * n_params <= PARAM_LIMIT
del model


## 데이터·증강

공통 크롭: 각 변 96~192픽셀, 정사각형 또는 직사각형(가로/세로 0.75~1.333).
직사각형은 192×192로 늘리며 비율을 변형합니다. 크롭 후 좌우 반전·색상 변형·원근을 적용하고 192×192로 맞춥니다.
색상 변형·가림·원근의 강도는 005의 같은 seed 설정을 유지합니다. 가림은 정규화 후 평균값 0으로 채웁니다.


In [ ]:
data_path = BASE_DIR / 'train'
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD
AUGMENTATION_SETTINGS = {
	'square': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.0},
	'rectangle': {'square': False, 'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.3, 'erase_p': 0.0, 'perspective_p': 0.0},
	'square_erasing': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.1, 'perspective_p': 0.0},
	'square_perspective': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.3},
}
train_transforms = {}
for name, settings in AUGMENTATION_SETTINGS.items():
	operations = [
		model_module.RandomPhotoCrop(square=settings['square']),
		transforms.RandomHorizontalFlip(p=0.5),
		transforms.ColorJitter(brightness=settings['brightness'], contrast=settings['contrast'], saturation=settings['saturation']),
	]
	if settings['perspective_p'] > 0:
		operations.append(transforms.RandomPerspective(distortion_scale=0.15, p=settings['perspective_p'], fill=model_module.PERSPECTIVE_FILL))
	operations.extend([
		transforms.Resize((IMG_SIZE, IMG_SIZE)),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])
	if settings['erase_p'] > 0:
		operations.append(transforms.RandomErasing(p=settings['erase_p'], scale=(0.02, 0.1), value=0))
	train_transforms[name] = transforms.Compose(operations)
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transforms['square'])
full_valid = datasets.ImageFolder(data_path, valid_transform)
assert full_train.class_to_idx == {'6e': 0, 'it': 1}
assert len(full_train) == 2906, '학습' + f' 사진 수가 다릅니다: {len(full_train)}'
for path, label in full_train.samples:
	with Image.open(path) as image:
		assert min(image.size) >= IMG_SIZE, f'192보다 작은 이미지: {path}'
print('클래스:', full_train.class_to_idx, '/ 전체:', len(full_train), '/ 입력:', IMG_SIZE)


## 층화 4-fold 분할

split seed 42로 한 번 나눕니다. 네 증강 모두 동일한 fold 인덱스를 사용합니다.
모든 사진은 정확히 한 번 검증셋에 들어갑니다. fold별 검증 726~727장·학습 2,179~2,180장입니다.


In [ ]:
labels = np.array([label for _, label in full_train.samples])
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_indices = list(splitter.split(np.arange(len(labels)), labels))
fold_assignments = np.zeros(len(labels), dtype=np.int64)
valid_counts = np.zeros(len(labels), dtype=np.int64)
for fold, (train_idx, valid_idx) in enumerate(fold_indices, start=1):
	assert not set(train_idx).intersection(valid_idx)
	assert len(train_idx) + len(valid_idx) == len(labels)
	fold_assignments[valid_idx] = fold
	valid_counts[valid_idx] += 1
	print(f'fold {fold} / seed {FOLD_SEEDS[fold - 1]} / train {len(train_idx)} / valid {len(valid_idx)}')
assert bool((valid_counts == 1).all()), '검증 사진 중복 또는 누락'
with (RESULT_DIR / 'fold_assignments.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'label', 'fold', 'seed'))
	writer.writerows((Path(path).stem, label, int(fold), FOLD_SEEDS[int(fold) - 1]) for (path, label), fold in zip(full_valid.samples, fold_assignments))


In [ ]:
preview_train_idx, preview_valid_idx = fold_indices[0]
dataloaders = {
	'train': DataLoader(Subset(full_train, preview_train_idx), batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(FOLD_SEEDS[0])),
	'valid': DataLoader(Subset(full_valid, preview_valid_idx), batch_size=params['batch_size'], shuffle=False, num_workers=0),
}
print('fold 1 미리보기 배치 수:', len(dataloaders['train']), len(dataloaders['valid']))


### 데이터 확인

In [ ]:
def img_show(tensor, title=None):
	arr = tensor.numpy().transpose((1, 2, 0))
	arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
	arr = np.clip(arr, 0, 1)
	plt.figure(figsize=(12, 4))
	plt.imshow(arr)
	plt.axis('off')
	if title is not None:
		plt.title(title)
	plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])


## 4-fold × 네 증강 — 총 16회 새 학습

매 fold에서 네 모델에 같은 seed를 적용합니다. 모델·데이터 로더·optimizer는 매번 새로 생성합니다.
각 모델은 ImageNet 가중치에서 전체 미세조정합니다. 기존 체크포인트를 재사용하지 않습니다.
학습률·체크포인트 선택·epoch CSV 저장은 011과 같습니다. fold/증강을 파일명에 포함해 중복 seed 파일 충돌을 방지합니다.
학습 완료 후 최저 검증 손실 가중치로 해당 fold 검증 사진에만 원본·반전 추론을 수행해 OOF 배열에 저장합니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
	"""optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
	training = optimizer is not None
	model.train() if training else model.eval()

	total_loss, total_correct, total_n = 0.0, 0, 0
	with torch.set_grad_enabled(training):
		for inputs, targets in loader:
			inputs = inputs.to(device)
			targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

			logits = model(inputs)
			loss = criterion(logits, targets)
			assert bool(torch.isfinite(loss)), "학습/검증 손실이 비유한 값입니다."

			if training:
				optimizer.zero_grad()
				loss.backward()
				optimizer.step()

			probs = torch.sigmoid(logits)
			preds = (probs >= 0.5).float()
			total_loss += loss.item() * inputs.size(0)
			total_correct += (preds == targets).sum().item()
			total_n += inputs.size(0)

	return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates):
	assert min_epochs > 0
	assert patience > 0
	assert len(learning_rates) > 0
	assert all(rate > 0 for rate in learning_rates)
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}', flush=True)

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
results = []
oof_members = torch.full((len(full_valid), len(AUGMENTATIONS)), float('nan'))
oof_tta_members = torch.full_like(oof_members, float('nan'))
oof_counts = np.zeros((len(full_valid), len(AUGMENTATIONS)), dtype=np.int64)
for filename in ('baseline_test6.ipynb', 'model_test6.py'):
	shutil.copy2(BASE_DIR / filename, CODE_DIR / filename)
source_hashes = {filename: hashlib.sha256((CODE_DIR / filename).read_bytes()).hexdigest() for filename in ('baseline_test6.ipynb', 'model_test6.py')}
training_record = {
	'experiment': '012', 'cv': 'StratifiedKFold', 'split_seed': SEED, 'n_splits': N_SPLITS,
	'fold_seeds': list(FOLD_SEEDS), 'seed_policy': 'one common seed for all four augmentations within each fold',
	'augmentations': list(AUGMENTATIONS), 'settings': AUGMENTATION_SETTINGS,
	'crop_side_range': [96, 192], 'rectangle_ratio_range': [0.75, 4 / 3],
	'preprocessing': 'center_crop_192_no_padding', 'params': params,
	'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
	'device': str(device), 'tf32': False, 'source_sha256': source_hashes,
}
(RESULT_DIR / 'training_record.json').write_text(json.dumps(training_record, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
for fold, (train_idx, valid_idx) in enumerate(fold_indices, start=1):
	seed = FOLD_SEEDS[fold - 1]
	fold_dir = RESULT_DIR / f'fold_{fold}'
	assert bool((fold_assignments[valid_idx] == fold).all())
	assert not set(train_idx).intersection(valid_idx)
	for augmentation_index, augmentation in enumerate(AUGMENTATIONS):
		full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
		assert full_run_train.samples == full_train.samples
		print(f'\n학습 {len(results) + 1}/{N_SPLITS * len(AUGMENTATIONS)} / fold {fold} / seed {seed} / {augmentation}', flush=True)
		seed_everything(seed)
		model = model_module.build_model(pretrained=True).to(device)
		seed_everything(seed)
		dataloaders = {
			'train': DataLoader(Subset(full_run_train, train_idx), batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed)),
			'valid': DataLoader(Subset(full_valid, valid_idx), batch_size=params['batch_size'], shuffle=False, num_workers=0),
		}
		criterion = nn.BCEWithLogitsLoss()
		optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rates'][0])
		pt_file_name = fold_dir / 'checkpoints' / f'efficientnet_b0_{augmentation}_seed{seed}.pt'
		assert not pt_file_name.exists(), f'기존 가중치를 덮어쓰지 않습니다: {pt_file_name}'
		started = time.perf_counter()
		with (fold_dir / 'epochs' / f'{augmentation}_seed{seed}.csv').open('w', newline='', encoding='utf-8-sig') as epoch_csv:
			history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params['epoch'], epoch_csv=epoch_csv, early_stopping=params['early_stopping'], min_epochs=params['min_epochs'], patience=params['patience'], learning_rates=params['learning_rates'])
		elapsed = time.perf_counter() - started
		model.load_state_dict(torch.load(pt_file_name, map_location=device, weights_only=True))
		model.eval()
		member_batches, tta_batches, target_batches = [], [], []
		with torch.inference_mode():
			for inputs, targets in dataloaders['valid']:
				inputs = inputs.to(device)
				probs = torch.sigmoid(model(inputs)).squeeze(1)
				flipped = torch.sigmoid(model(inputs.flip(dims=[3]))).squeeze(1)
				member_batches.append(probs.cpu())
				tta_batches.append(((probs + flipped) / 2).cpu())
				target_batches.append(targets)
		assert torch.equal(torch.cat(target_batches), torch.as_tensor(labels[valid_idx]))
		assert bool((oof_counts[valid_idx, augmentation_index] == 0).all())
		oof_members[valid_idx, augmentation_index] = torch.cat(member_batches)
		oof_tta_members[valid_idx, augmentation_index] = torch.cat(tta_batches)
		oof_counts[valid_idx, augmentation_index] += 1
		np.savez_compressed(RESULT_DIR / 'oof_partial.npz', members=oof_members.numpy(), tta_members=oof_tta_members.numpy(), counts=oof_counts, folds=fold_assignments)
		results.append({
			'fold': fold, 'model': 'efficientnet_b0', 'seed': seed, 'augmentation': augmentation,
			'train_samples': len(train_idx), 'valid_samples': len(valid_idx), 'image_size': IMG_SIZE,
			'selection_metric': 'valid_loss', 'learning_rates': str(params['learning_rates']),
			'max_epochs': params['epoch'], 'epochs_run': len(history['train_loss']),
			'early_stopping': params['early_stopping'], 'min_epochs': params['min_epochs'], 'patience': params['patience'],
			'best_epoch': best_epoch, 'valid_loss': best_loss, 'valid_acc': history['valid_acc'][best_epoch - 1],
			'seconds': elapsed, 'n_params': n_params, 'checkpoint': str(pt_file_name),
		})
		with (RESULT_DIR / 'runs.csv').open('w', newline='', encoding='utf-8-sig') as output:
			writer = csv.DictWriter(output, fieldnames=list(results[0]))
			writer.writeheader()
			writer.writerows(results)
		epochs = range(1, len(history['train_loss']) + 1)
		fig, axes = plt.subplots(1, 2, figsize=(12, 4))
		for axis, metric in zip(axes, ('loss', 'acc')):
			axis.plot(epochs, history[f'train_{metric}'], label='train')
			axis.plot(epochs, history[f'valid_{metric}'], label='valid')
			axis.set_xlabel('epoch')
			axis.set_ylabel(metric)
			axis.legend()
			axis.grid(alpha=0.3)
		fig.suptitle(f'fold {fold} / seed {seed} / {augmentation} / selected epoch {best_epoch}')
		fig.tight_layout()
		fig.savefig(fold_dir / 'graphs' / f'{augmentation}_seed{seed}.png', dpi=140)
		plt.close(fig)
		del model, optimizer, dataloaders
		if device.type == 'cuda':
			torch.cuda.empty_cache()
assert len(results) == N_SPLITS * len(AUGMENTATIONS) == 16
assert bool((oof_counts == 1).all()), 'OOF 예측 중복 또는 누락'
assert bool(torch.isfinite(oof_members).all()) and bool(torch.isfinite(oof_tta_members).all())


## OOF 전체 성능·fold별 성능

각 사진의 네 예측은 모두 그 사진을 학습하지 않은 같은 fold 모델에서 나왔습니다.
다른 fold 모델을 같은 검증 사진에 추가하지 않습니다. 전체 OOF와 fold별 지표를 함께 기록합니다.
체크포인트 선택에 해당 fold 검증셋을 사용했으므로 완전히 별도인 테스트 성능과 구분합니다.


In [ ]:
oof_targets = torch.as_tensor(labels, dtype=torch.float32)
oof_outputs = {augmentation: oof_members[:, index] for index, augmentation in enumerate(AUGMENTATIONS)}
oof_outputs['mean_T1'] = oof_members.mean(dim=1)
oof_outputs['T0.3'] = model_module.combine_probabilities(oof_members, 'T0.3').squeeze(1)
oof_outputs['T0.5'] = model_module.combine_probabilities(oof_members, 'T0.5').squeeze(1)
oof_outputs['T0.3_TTA'] = model_module.combine_probabilities(oof_tta_members, 'T0.3').squeeze(1)
oof_metrics, fold_metrics = [], []
for fold in range(N_SPLITS + 1):
	mask = torch.ones(len(labels), dtype=torch.bool) if fold == 0 else torch.as_tensor(fold_assignments == fold)
	for name, probs in oof_outputs.items():
		selected_probs, targets = probs[mask], oof_targets[mask]
		errors = int(((selected_probs >= 0.5) != targets.bool()).sum())
		loss = torch.nn.functional.binary_cross_entropy(selected_probs.clamp(1e-6, 1 - 1e-6), targets).item()
		row = {'fold': fold, 'model': name, 'samples': len(targets), 'errors': errors, 'valid_acc': 1 - errors / len(targets), 'valid_loss': loss}
		if fold == 0:
			oof_metrics.append(row)
			print(f'OOF {name}: {len(targets)}장 / 오답 {errors} / Log Loss {loss:.8f}')
		else:
			fold_metrics.append(row)
for filename, rows in (('oof_metrics.csv', oof_metrics), ('fold_metrics.csv', fold_metrics)):
	with (RESULT_DIR / filename).open('w', newline='', encoding='utf-8-sig') as output:
		writer = csv.DictWriter(output, fieldnames=list(rows[0]))
		writer.writeheader()
		writer.writerows(rows)
with (RESULT_DIR / 'oof_predictions.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(['id', 'label', 'fold', 'seed'] + [f'{name}_prob' for name in AUGMENTATIONS] + [f'{name}_tta_prob' for name in AUGMENTATIONS] + ['T0.3', 'T0.5', 'T0.3_TTA'])
	for index, ((path, label), fold) in enumerate(zip(full_valid.samples, fold_assignments)):
		writer.writerow([Path(path).stem, label, int(fold), FOLD_SEEDS[int(fold) - 1]] + oof_members[index].tolist() + oof_tta_members[index].tolist() + [float(oof_outputs[name][index]) for name in ('T0.3', 'T0.5', 'T0.3_TTA')])
agreement_rows = []
for name, members in (('T0.3', oof_members), ('T0.3_TTA', oof_tta_members)):
	votes = (members >= 0.5).sum(dim=1)
	for agreement, mask in (('4:0', (votes == 0) | (votes == 4)), ('3:1', (votes == 1) | (votes == 3)), ('2:2', votes == 2)):
		count = int(mask.sum())
		errors = int(((oof_outputs[name][mask] >= 0.5) != oof_targets[mask].bool()).sum())
		loss = torch.nn.functional.binary_cross_entropy(oof_outputs[name][mask].clamp(1e-6, 1 - 1e-6), oof_targets[mask]).item() if count else None
		agreement_rows.append({'rule': name, 'agreement': agreement, 'samples': count, 'errors': errors, 'valid_loss': loss})
with (RESULT_DIR / 'agreement_metrics.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.DictWriter(output, fieldnames=list(agreement_rows[0]))
	writer.writeheader()
	writer.writerows(agreement_rows)
print('OOF·fold별 성능 저장 완료:', RESULT_DIR)


## 제출 fold 선택 — 여기부터는 재학습 없이 실행 가능

SUBMISSION_FOLD 기본값은 1입니다. 다른 fold를 내보내려면 값을 1~4 중 하나로 바꾸고 이 아래 셀만 실행합니다.
전체 16모델을 평균하지 않습니다. 선택한 fold의 네 증강 모델만 사용하므로 20M 이하입니다.
OOF는 4-fold 전체 검증 결과이며 선택한 fold 하나의 테스트 성능을 보장하지 않습니다.


In [ ]:
SUBMISSION_FOLD = 1
assert 1 <= SUBMISSION_FOLD <= N_SPLITS
selected_seed = FOLD_SEEDS[SUBMISSION_FOLD - 1]
SUBMISSION_TAG = f'fold{SUBMISSION_FOLD}_seed{selected_seed}'
SUBMISSION_DIR = RESULT_DIR / f'submission_fold_{SUBMISSION_FOLD}'
assert not (SUBMISSION_DIR / f'submission_{SUBMISSION_TAG}_T0.3.csv').exists(), f'기존 CSV를 덮어쓰지 않습니다: {SUBMISSION_DIR}'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
selected_checkpoint_dir = RESULT_DIR / f'fold_{SUBMISSION_FOLD}' / 'checkpoints'
checkpoint = {
	'experiment': '012', 'n_splits': N_SPLITS, 'fold': SUBMISSION_FOLD, 'seed': selected_seed,
	'image_size': IMG_SIZE, 'preprocessing': 'center_crop_192_no_padding',
	'seeds': [selected_seed] * len(AUGMENTATIONS), 'augmentations': list(AUGMENTATIONS),
	'state_dicts': [torch.load(selected_checkpoint_dir / f'efficientnet_b0_{name}_seed{selected_seed}.pt', map_location='cpu', weights_only=True) for name in AUGMENTATIONS],
}
torch.save(checkpoint, SUBMISSION_DIR / 'ensemble.pt')
torch.set_num_threads(4)
ensemble = model_module.EfficientNetEnsemble(checkpoint['state_dicts']).eval()
ensemble_params = sum(parameter.numel() for parameter in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT
print(f'제출 선택: fold {SUBMISSION_FOLD} / seed {selected_seed} / {ensemble_params:,} parameters')

class TestImageDataset(Dataset):
	def __init__(self, files, transform):
		self.files = files
		self.transform = transform

	def __len__(self):
		return len(self.files)

	def __getitem__(self, index):
		path = self.files[index]
		with Image.open(path) as image:
			return self.transform(image.convert('RGB')), path.stem


test_files = sorted(path for path in (BASE_DIR / 'test').iterdir() if path.is_file() and (path.suffix.lower() in ('.jpg', '.jpeg', '.png')))
with (BASE_DIR / 'sample_submission.csv').open(newline='', encoding='utf-8-sig') as source:
	sample_rows = list(csv.DictReader(source))
expected_ids = {row['id'] for row in sample_rows}
assert (len(test_files) == 500) and (len(expected_ids) == 500)
assert len({path.stem for path in test_files}) == 500
assert {path.stem for path in test_files} == expected_ids
test_loader = DataLoader(TestImageDataset(test_files, valid_transform), batch_size=params['batch_size'], shuffle=False, num_workers=0)


In [ ]:
submission_rows = {name: [] for name in ('T0.3', 'T0.5', 'T0.3_TTA')}
test_member_rows = []
started = time.perf_counter()
with torch.inference_mode():
	for inputs, image_ids in test_loader:
		members = ensemble.member_probabilities(inputs)
		tta_members = (members + ensemble.member_probabilities(inputs.flip(dims=[3]))) / 2
		outputs = {
			'T0.3': model_module.combine_probabilities(members, 'T0.3').squeeze(1),
			'T0.5': model_module.combine_probabilities(members, 'T0.5').squeeze(1),
			'T0.3_TTA': model_module.combine_probabilities(tta_members, 'T0.3').squeeze(1),
		}
		for position, image_id in enumerate(image_ids):
			test_member_rows.append([image_id] + members[position].tolist() + tta_members[position].tolist())
		for name, probs in outputs.items():
			assert bool(torch.isfinite(probs).all())
			submission_rows[name].extend((image_id, f'{prob.item():.6f}') for image_id, prob in zip(image_ids, probs))
test_cpu_seconds = time.perf_counter() - started
for name, rows in submission_rows.items():
	path = SUBMISSION_DIR / f'submission_{SUBMISSION_TAG}_{name}.csv'
	with path.open('w', newline='', encoding='utf-8') as output:
		writer = csv.writer(output)
		writer.writerow(('id', 'prob'))
		writer.writerows(rows)
	assert (len(rows) == 500) and ({row[0] for row in rows} == expected_ids)
	assert all(0 <= float(row[1]) <= 1 for row in rows)
	print('CSV:', path)
with (SUBMISSION_DIR / 'test_member_probabilities.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(['id'] + [f'{name}_prob' for name in AUGMENTATIONS] + [f'{name}_tta_prob' for name in AUGMENTATIONS])
	writer.writerows(test_member_rows)
print(f'CPU 원본+반전 추론: {test_cpu_seconds:.2f}s / 500장')


## 선택 fold의 제출 ZIP 생성

submission_fold_N/에 세 CSV와 각 대응 ZIP을 생성합니다. CSV와 같은 규칙 이름의 ZIP을 제출합니다.
각 ZIP 기본 명령이 해당 temperature·TTA 규칙을 재현합니다. 선택한 네 가중치만 포함하며 나머지 fold 모델은 추론하지 않습니다.


In [ ]:
%%writefile predict_test6.py
import argparse
import csv
from pathlib import Path

import torch
from PIL import Image

from model_test6 import IMG_SIZE, TEMPERATURE_RULES, EfficientNetEnsemble, build_valid_transform, combine_probabilities

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
	files = sorted(path for path in Path(input_dir).iterdir() if path.is_file() and (path.suffix.lower() in (".jpg", ".jpeg", ".png")))
	assert files, f"이미지가 없습니다: {input_dir}"
	assert len({path.stem for path in files}) == len(files), "이미지 ID가 중복됩니다."
	return files


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--temperature-rule", choices=TEMPERATURE_RULES, default="T0.3")
	parser.add_argument("--tta", action="store_true", help="모델별 원본·좌우 반전 확률 평균 후 결합")
	args = parser.parse_args()
	torch.set_num_threads(4)
	weights_path = Path(args.weights)
	if not weights_path.is_absolute():
		weights_path = Path(__file__).resolve().parent / weights_path
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE
	assert checkpoint["preprocessing"] == "center_crop_192_no_padding"
	assert checkpoint["experiment"] == "012"
	assert checkpoint["n_splits"] == 4
	assert 1 <= checkpoint["fold"] <= 4
	assert checkpoint["seed"] == (42, 43, 44, 45)[checkpoint["fold"] - 1]
	assert checkpoint["seeds"] == [checkpoint["seed"]] * 4
	assert checkpoint["augmentations"] == ["square", "rectangle", "square_erasing", "square_perspective"]
	model = EfficientNetEnsemble(checkpoint["state_dicts"]).eval()
	n_params = sum(parameter.numel() for parameter in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	transform = build_valid_transform()
	files = load_images(args.input)
	rows = []
	with torch.inference_mode():
		for position in range(0, len(files), 32):
			batch_files = files[position:position + 32]
			images = []
			for path in batch_files:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images)
			member_probs = model.member_probabilities(inputs)
			if args.tta:
				member_probs = (member_probs + model.member_probabilities(inputs.flip(dims=[3]))) / 2
			probs = combine_probabilities(member_probs, args.temperature_rule).squeeze(1)
			assert bool(torch.isfinite(probs).all())
			for path, prob in zip(batch_files, probs):
				rows.append((path.stem, f"{prob.item():.6f}"))
	with Path(args.output).open("w", newline="", encoding="utf-8") as output:
		writer = csv.writer(output)
		writer.writerow(("id", "prob"))
		writer.writerows(rows)
	print(f"{args.output} 저장 완료 / {len(files)}장 / {n_params:,} parameters / TTA={args.tta}")


if __name__ == "__main__":
	main()


In [ ]:
assert hashlib.sha256((BASE_DIR / 'model_test6.py').read_bytes()).hexdigest() == source_hashes['model_test6.py'], '학습 후 모델 코드를 바꾸지 마세요.'
shutil.copy2(BASE_DIR / 'predict_test6.py', CODE_DIR / 'predict_test6.py')
package_paths = {}
submission_hashes = {name: hashlib.sha256((SUBMISSION_DIR / f'submission_{SUBMISSION_TAG}_{name}.csv').read_bytes()).hexdigest() for name in submission_rows}
for name in submission_rows:
	package_dir = SUBMISSION_DIR / f'package_{name}'
	assert not package_dir.exists(), f'기존 패키지를 덮어쓰지 않습니다: {package_dir}'
	package_dir.mkdir()
	shutil.copy2(CODE_DIR / 'model_test6.py', package_dir / 'model.py')
	shutil.copy2(CODE_DIR / 'baseline_test6.ipynb', package_dir / 'baseline_test6.ipynb')
	shutil.copy2(SUBMISSION_DIR / 'ensemble.pt', package_dir / 'model.pt')
	shutil.copy2(RESULT_DIR / 'training_record.json', package_dir / 'training_record.json')
	shutil.copy2(RESULT_DIR / 'fold_assignments.csv', package_dir / 'fold_assignments.csv')
	source = (CODE_DIR / 'predict_test6.py').read_text(encoding='utf-8').replace('from model_test6 import ', 'from model import ')
	if name == 'T0.5':
		source = source.replace('default="T0.3"', 'default="T0.5"')
	if name == 'T0.3_TTA':
		source = source.replace('action="store_true"', 'action="store_true", default=True')
	(package_dir / 'predict.py').write_text(source, encoding='utf-8')
	readme = f'''실험 012 / CSV: submission_{SUBMISSION_TAG}_{name}.csv
실행: python predict.py --input <test 폴더> --output pred.csv
기본 규칙: {name}. 추가 옵션 없이 대응 CSV를 재현합니다.
전처리: 원본 중앙 192x192 크롭, 패딩 없음, ImageNet 정규화.
TTA: {'원본·좌우 반전 모델별 확률 평균' if name.endswith('TTA') else '미적용'}.
기본 T0.3: 전원 일치 평균/T=0.3, 불일치 가운데 두 확률 평균/T=0.7.
T0.5: 같은 결합 확률에 전체 T=0.5.
학습: 층화 4-fold × 네 증강, 총 16회 새 학습. 이 ZIP은 fold {SUBMISSION_FOLD}의 네 모델만 사용. 공통 seed {selected_seed}.
전체 파라미터: {ensemble_params:,}. 추론 중 다운로드 없음.
학습 코드: baseline_test6.ipynb. 학습 재현 시 train/test/sample_submission.csv를 준비하고 루트에서 실행.
Fold 배정: fold_assignments.csv. 전체 OOF는 CV 지표이며 선택 fold 하나의 성적이 아닙니다.
CSV SHA-256: {submission_hashes[name]}
필요 모듈: Python 3.10+, torch, torchvision, Pillow. 학습에는 numpy/scikit-learn/matplotlib 추가.
학습 환경: {sys.version.split()[0]}, torch {torch.__version__}, torchvision {torchvision.__version__}.
'''
	(package_dir / 'README.txt').write_text(readme, encoding='utf-8')
	zip_path = Path(shutil.make_archive(str(SUBMISSION_DIR / f'submission_package_{name}'), 'zip', package_dir))
	assert zip_path.stat().st_size <= 200_000_000, 'ZIP 200MB 초과'
	with zipfile.ZipFile(zip_path) as archive:
		assert archive.testzip() is None
	package_paths[name] = package_dir
	print('ZIP:', zip_path, f'({zip_path.stat().st_size:,} bytes)')


## 실제 ZIP CPU 재현 검사

각 ZIP을 풀어 기본 명령으로 500장 CSV를 재현합니다. 최대 차이 0.00001 이하·실행별 600초 이하를 검사합니다.
선택한 submission_fold_N/package_check.json에 기록합니다. 로컬 CPU 시간은 실제 Colab CPU 시간과 구분합니다.


In [ ]:
def read_probs(path):
	with Path(path).open(newline='', encoding='utf-8-sig') as source:
		reader = csv.DictReader(source)
		assert reader.fieldnames == ['id', 'prob']
		rows = list(reader)
	assert len(rows) == 500
	assert len({row['id'] for row in rows}) == 500
	assert {row['id'] for row in rows} == expected_ids
	assert all(np.isfinite(float(row['prob'])) and (0 <= float(row['prob']) <= 1) for row in rows)
	return {row['id']: float(row['prob']) for row in rows}


checks = {}
for name in package_paths:
	zip_path = SUBMISSION_DIR / f'submission_package_{name}.zip'
	verify_dir = SUBMISSION_DIR / f'package_verify_{name}'
	assert not verify_dir.exists(), f'검증 폴더가 이미 있습니다: {verify_dir}'
	with zipfile.ZipFile(zip_path) as archive:
		archive.extractall(verify_dir)
	output_path = SUBMISSION_DIR / f'package_check_{SUBMISSION_TAG}_{name}.csv'
	started = time.perf_counter()
	process = subprocess.run([sys.executable, 'predict.py', '--input', str(BASE_DIR / 'test'), '--output', str(output_path)], cwd=verify_dir, capture_output=True, text=True, encoding='utf-8', errors='strict', timeout=600, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
	seconds = time.perf_counter() - started
	print(process.stdout)
	assert process.returncode == 0, process.stderr
	assert seconds <= 600, f'CPU 600초 초과: {seconds:.2f}'
	original = read_probs(SUBMISSION_DIR / f'submission_{SUBMISSION_TAG}_{name}.csv')
	reproduced = read_probs(output_path)
	max_diff = max(abs(original[image_id] - reproduced[image_id]) for image_id in original)
	assert max_diff <= 0.00001, f'CSV 재현 차이: {max_diff}'
	checks[name] = {'seconds_local_cpu': seconds, 'max_probability_difference': max_diff, 'csv_sha256': submission_hashes[name], 'zip_sha256': hashlib.sha256(zip_path.read_bytes()).hexdigest()}
	print(f'{name}: CPU {seconds:.2f}s / 최대 차이 {max_diff:.8f} / 통과')
(SUBMISSION_DIR / 'package_check.json').write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding='utf-8')
print('제출 준비 완료:', SUBMISSION_DIR)


## 결과 확인·제출

1. 전체 성능: oof_metrics.csv. fold별 성능: fold_metrics.csv. 사진별 예측: oof_predictions.csv.
2. 선택 fold의 submission_fold_N/submission_foldN_seedS_*.csv를 제출합니다.
3. 같은 규칙 이름의 submission_package_*.zip을 최종 제출합니다.
4. 다른 fold를 내보낼 때는 SUBMISSION_FOLD만 바꾸고 선택 셀부터 아래만 실행합니다. 학습 셀을 다시 실행하지 않습니다.

fold마다 학습 데이터와 seed가 함께 다르므로 fold 간 차이를 seed 하나의 효과로 분리하지 않습니다.
같은 fold의 네 모델에 같은 seed를 적용해도 증강별 난수 소비가 달라, 모든 이미지 변형까지 동일하지는 않습니다.
